### Sentiment Analysis with an Recurrent Neural Network(RNN):
- Recurrent Neural Networks(RNNs) are used in sequence tasks such as sentiment analysis due to their ability to capture context from sequential data.
- In this article we apply RNNs to analyze the sentiment of customer reviews from Swiggy food delivery platform.
- The goal is to classify reviews as positive or negative for providing insights into customer experiences.
#### 1. Importing Required Libraries:


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

C:\Users\techs\anaconda3\New folder\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


#### 2. Load Dataset:
- pd.read_csv - Reads the CSV file into a Pandas DataFrame.
- data.columns - Accesses the column names of the DataFrame.
- tolist() - Converts the column names from an index object to a regular Python list.

In [2]:
data = pd.read_csv('swiggy.csv')
print("Columns in the dataset:")
print(data.columns.tolist())

Columns in the dataset:
['ID', 'Area', 'City', 'Restaurant Price', 'Avg Rating', 'Total Rating', 'Food Item', 'Food Type', 'Delivery Time', 'Review']


#### 3. Text Cleaning and Sentiment Labeling:
- We will clean the review text, create a sentiment label based on ratings and remove any missing values.
- data['Review'] = data['Review'].str.lower() - Converts all text in the "Reivew" column to lowercase.
- data['Review'] = data['Review'].replace(r'[^a-z0-9\s]',' ', regexTrue) - Removes all characters except letters, numbers and soaces from the "Review" column.
- data['sentiment'] = data['Avg Rating'].apply(lambda x: 1 if x>3.5 else 0) - Creates a new "setiment" column with 1 for ratings above 3.5 and 0 otherwise.
- data = data.dropna() - Removes rows that contain any missing values.

In [3]:
data['Review'] = data['Review'].str.lower()
data['Review'] = data['Review'].replace(r'[a-z0-9\s]', ' ', regex=True)

data['sentiment'] = data['Avg Rating'].apply(lambda x: 1 if x>3.5 else 0)
data = data.dropna()

#### 4. Tokenization and Padding:
- We will prepare the text data by tokenizing and padding it and extract the target sentiment labels.
- Tokenizer converts words into integer sequences and padding esures all input sequences have the same length(max_length).
  - max_features=5000 - Sets the maximum number of words to keep in the tokeizer.
  - max_length=200 - Defines the fixed length for each input sequence after padding.
  - Tokenizer(num_words=max_features) - Initializes the tokenizer to keep the top 5000 words only.
  - tokenizer.fit_on_text(data['Review']) - Builds the word index based on the reviews in the dataset.
  - tokenizer.texts_to_sequences(data['Review']) - Converts each review into a sequence of word indexes.
  - pad_sequences(..., maxlen=max_legth) - Pads or truncates each sequences to the same length.
  - y = data['sentiment'].values - Extracts the sentiment labels as a Numpy array for model training.

In [4]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data['Review'])

X = pad_sequences(tokenizer.texts_to_sequences(data['Review']), maxlen=max_length)
y = data['sentiment'].values

#### 5. Splitting the Data:
- train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) - Splits data into 80% training and 20% test sets, preserving sentiment class balance.
- train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train) - Further splis training data into 90% training and 10% validation sets, keeping class distribution consistent. 

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, 
                                                    stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, 
                                                  random_state=42, stratify=y_train)


#### 6. Building RNN Model:
- We will build and compile a simple RNN model for binary sentiment classification.
  - Sequential([...]) - Creates a sequential neural network model.
  - Embedding(input_dim=max_features, output_dim=16, input_length=max_length) - Maps input words to 16-dimensional vectors.
  - SimpleRNN(64, activation='tanh', return_sequences=False) - Adds a recurrent layer with 64 units using tanh activation.
  - Dense(1, activation='sigmoid') - Adds an output layer with one neuron using sigmoid activation for binary output.
    

In [9]:
model = Sequential([
    Embedding(input_dim=max_features, output_dim=16, input_length=max_length),
    SimpleRNN(64, activation='tanh', return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#### 7. Training and Evaluating Model:
- We will train the model on training data, validate it during trining, then evaluate its performance on test data.
  - model.fit(...) - Trains the model for 5 epochs with batchsize 32, validating on the validation set.
  - model.evaluate(X_test, y_test, verbose=0) - Evaluates the trained model on test data without extra output.
  

In [10]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, 
                    validation_data=(X_val, y_val), verbose=1)
score = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy:{score[1]:.2f}')

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.7160 - loss: 0.6002 - val_accuracy: 0.7156 - val_loss: 0.5971
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7160 - loss: 0.5990 - val_accuracy: 0.7156 - val_loss: 0.5987
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7160 - loss: 0.5986 - val_accuracy: 0.7156 - val_loss: 0.5985
Epoch 4/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7160 - loss: 0.5986 - val_accuracy: 0.7156 - val_loss: 0.5976
Epoch 5/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7160 - loss: 0.5981 - val_accuracy: 0.7156 - val_loss: 0.5989
Test Accuracy:0.72


#### 8. Predicting Sentiment:
- We will create a function to preprocess a single review, predict its sentiment and display the result.
  - review_text.lower() - Converts the input review text to lowercase.
  - re.sub(r'[a-z0-9\s]', ' ', text) - Removes all characters except letters, numbers, and spaces.
  - tokenizer.texts_to_sequences([text]) - Converts the cleaned review into a sequence of word indexes.
  - pad_sequences(seq, maxlen=max_length) - Pads the sequence to the fixed length
  - model.predict(padded)[0][0] - Predicts the sentiment probability for the review.
  - Returns "Positive" if prediction is 0.5 or above, otherwise "Negative" including the probability score.

In [11]:
def predict_sentiment(review_text):
    text = review_text.lower()
    text = re.sub(r'[a-z0-9\s]', ' ',text)
    
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    return f"{'Positive' if prediction >=0.5 else 'Negative'} (Probability:{prediction:.2f})"

sample_review = "The food was great"
print(f"Review:{sample_review}")
print(f"Sentiment:{predict_sentiment(sample_review)}")

Review:The food was great
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
Sentiment:Positive (Probability:0.69)
